In [1]:
import pandas as pd
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import warnings
warnings.filterwarnings("ignore")

In [2]:
import pyodbc

conn_str = (
    "Driver={ODBC Driver 18 for SQL Server};"
    "Server=localhost;"
    "Database=MarketingAnalytics;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

conn = pyodbc.connect(conn_str)

print("Connection successful!")


Connection successful!


In [26]:
query="select * from dbo.customer_reviews;"

In [27]:
query

'select * from dbo.customer_reviews;'

In [28]:
df = pd.read_sql(query, conn)
df.head()


,ReviewID,CustomerID,ProductID,ReviewDate,Rating,ReviewText
0,1,77,18,2023-12-23,3,"Average experience, nothing special."
1,2,80,19,2024-12-25,5,The quality is top-notch.
2,3,50,13,2025-01-26,4,Five stars for the quick delivery.
3,4,78,15,2025-04-21,3,"Good quality, but could be cheaper."
4,5,64,2,2023-07-16,3,"Average experience, nothing special."


In [29]:
df.head()

,ReviewID,CustomerID,ProductID,ReviewDate,Rating,ReviewText
0,1,77,18,2023-12-23,3,"Average experience, nothing special."
1,2,80,19,2024-12-25,5,The quality is top-notch.
2,3,50,13,2025-01-26,4,Five stars for the quick delivery.
3,4,78,15,2025-04-21,3,"Good quality, but could be cheaper."
4,5,64,2,2023-07-16,3,"Average experience, nothing special."


In [44]:
# Fumction to connect data from 
def fetch_data_from_sql():
    conn_str = ("Driver={ODBC Driver 18 for SQL Server};"
      "Server=localhost;"
      "Database=MarketingAnalytics;"
      "Trusted_Connection=yes;"
      "TrustServerCertificate=yes;")
    conn = pyodbc.connect(conn_str)
    query="select * from dbo.customer_reviews;"
    df = pd.read_sql(query, conn)
    conn.close()
    return df




In [45]:
Customers_review_df=fetch_data_from_sql()

In [46]:
Customers_review_df

,ReviewID,CustomerID,ProductID,ReviewDate,Rating,ReviewText
0,1,77,18,2023-12-23,3,"Average experience, nothing special."
1,2,80,19,2024-12-25,5,The quality is top-notch.
2,3,50,13,2025-01-26,4,Five stars for the quick delivery.
3,4,78,15,2025-04-21,3,"Good quality, but could be cheaper."
4,5,64,2,2023-07-16,3,"Average experience, nothing special."
...,...,...,...,...,...,...
1358,1359,28,4,2023-05-25,3,Not worth the money.
1359,1360,58,12,2023-11-13,2,"Average experience, nothing special."
1360,1361,96,15,2023-03-07,5,Customer support was very helpful.
1361,1362,99,2,2025-12-03,1,Product did not meet my expectations.


In [47]:
nltk.download('vader_lexicon')

[nltk_data] Error loading vader_lexicon: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>


False

In [43]:
sia=SentimentIntensityAnalyzer()

In [35]:
# Function to assign sentiment score
def calculate_sentiment(review):
    sentiment=sia.polarity_scores(review)
    return sentiment["compound"] # normalize the score between -1(most negative) and 1(most positive)
   
    


In [48]:
Customers_review_df['SentimentScore']=Customers_review_df['ReviewText'].apply(calculate_sentiment)

In [49]:
Customers_review_df.head()

,ReviewID,CustomerID,ProductID,ReviewDate,Rating,ReviewText,SentimentScore
0,1,77,18,2023-12-23,3,"Average experience, nothing special.",-0.3089
1,2,80,19,2024-12-25,5,The quality is top-notch.,0.0000
2,3,50,13,2025-01-26,4,Five stars for the quick delivery.,0.0000
3,4,78,15,2025-04-21,3,"Good quality, but could be cheaper.",0.2382
4,5,64,2,2023-07-16,3,"Average experience, nothing special.",-0.3089


In [50]:
# sentiment score category based on rating and score
# positive , rating > 4 ,score > 0.05
# mixed positive rating ==3
# Negative , Rating <=2 , Score < -0.05
# mixed Negative == 3


In [51]:
def categorize_sentiment(score,rating):
    if score > 0.05:  # positive sentiment score
        if rating >=4:
            return "Positive"
        elif rating ==3:
            return "Mixed Positive"
        else:
            return "Mixed Negative"

    elif score < -0.05:  # Negative sentiment score
        if rating <=2:
            return "Negative"
        elif rating ==3:
            return "Mixed Negative"
        else:
            return "Mixed Positive"

    else:  # Neutral sentiment score
        if rating >= 4:
            return "Positive"
        elif rating <=2:
            return "Negative"
        else:
            return "Neutral"


In [52]:
Customers_review_df['SentimentCategory'] = Customers_review_df.apply(
    lambda row: categorize_sentiment(row["SentimentScore"], row["Rating"]),
    axis=1
)


In [53]:
Customers_review_df.head()

,ReviewID,CustomerID,ProductID,ReviewDate,Rating,ReviewText,SentimentScore,SentimentCategory
0,1,77,18,2023-12-23,3,"Average experience, nothing special.",-0.3089,Mixed Negative
1,2,80,19,2024-12-25,5,The quality is top-notch.,0.0000,Positive
2,3,50,13,2025-01-26,4,Five stars for the quick delivery.,0.0000,Positive
3,4,78,15,2025-04-21,3,"Good quality, but could be cheaper.",0.2382,Mixed Positive
4,5,64,2,2023-07-16,3,"Average experience, nothing special.",-0.3089,Mixed Negative


In [54]:
# Sentiment bucket : group your score into text
def Sentiment_Bucket(score):
    if score >0.05:
        return "0.5 to 0.1" # srongly positive segment
    elif 0.0<= score<0.05:
        return "0.0 to 0.49" # mildy positive
    elif -0.05 <= score<0.0:
        return "-0.49 to 0.0"
    else:
        return "-1.0 to -0.5" # Strongly Negativr

In [57]:
Customers_review_df['SentimentBucket'] = Customers_review_df['SentimentScore'].apply(Sentiment_Bucket)



In [58]:
Customers_review_df.head()

,ReviewID,CustomerID,ProductID,ReviewDate,Rating,ReviewText,SentimentScore,SentimentCategory,SentimentBucket
0,1,77,18,2023-12-23,3,"Average experience, nothing special.",-0.3089,Mixed Negative,-1.0 to -0.5
1,2,80,19,2024-12-25,5,The quality is top-notch.,0.0000,Positive,0.0 to 0.49
2,3,50,13,2025-01-26,4,Five stars for the quick delivery.,0.0000,Positive,0.0 to 0.49
3,4,78,15,2025-04-21,3,"Good quality, but could be cheaper.",0.2382,Mixed Positive,0.5 to 0.1
4,5,64,2,2023-07-16,3,"Average experience, nothing special.",-0.3089,Mixed Negative,-1.0 to -0.5


In [61]:
Customers_review_df['SentimentCategory'].value_counts()

SentimentCategory
Positive          840
Negative          226
Mixed Negative    196
Mixed Positive     86
Neutral            15
Name: count, dtype: int64

In [63]:
Customers_review_df.to_csv("C:/Users/user/OneDrive/Desktop/data analytics/Python/Customer_review\.Customers_review_sentiment.csv",index=False)